
**Preparación del Entorno e Importación de Librerías**


In [1]:
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

# Configurar visualización en Jupyter
%matplotlib inline


**Bloque OpenCV: Filtros Espaciales Avanzados**

In [ ]:
def opencv_spatial_processing(image_path):
    # 1. Leer imagen en escala de grises para análisis de bordes
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError("No se pudo cargar la imagen. Verifica la ruta.")

    # 2. Filtro Espacial 1: Desfoque Gaussiano (Reducción de ruido)
    # Kernel de 5x5 con desviación estándar automática (0)
    blurred_img = cv2.GaussianBlur(img, (5, 5), 0)

    # 3. Filtro Espacial 2: Detección de Bordes con Sobel (Gradiente compuesto)
    sobel_x = cv2.Sobel(blurred_img, cv2.CV_64F, 1, 0, ksize=3) # Gradiente X
    sobel_y = cv2.Sobel(blurred_img, cv2.CV_64F, 0, 1, ksize=3) # Gradiente Y

    # Magnitud del gradiente aproximada: sqrt(Gx^2 + Gy^2)
    sobel_combined = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_combined = np.uint8(np.clip(sobel_combined, 0, 255))

    return img, blurred_img, sobel_combined


**Bloque PyTorch: Pipeline Estocástico de Data Augmentation**

In [ ]:
def get_pytorch_augmentation_pipeline():
    # Definición de transformaciones aleatorias consecutivas
    pipeline = transforms.Compose([
        # 1. Redimensionar y recortar aleatoriamente (Escala entre 60% y 100%)
        transforms.RandomResizedCrop(size=(224, 224), scale=(0.6, 1.0)),

        # 2. Volteo horizontal con un 50% de probabilidad
        transforms.RandomHorizontalFlip(p=0.5),

        # 3. Alteración fotométrica: Brillo, contraste y saturación
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),

        # 4. Transformar a tensor de PyTorch (Escala automáticamente los píxeles a [0.0, 1.0])
        transforms.ToTensor(),

        # 5. Normalización estándar de ImageNet (Media y desviación estándar por canal)
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return pipeline

def apply_pytorch_pipeline(image_path, pipeline, num_samples=3):
    # PyTorch y PIL leen en formato RGB de forma nativa
    orig_img = Image.open(image_path).convert('RGB')
    augmented_samples = [pipeline(orig_img) for _ in range(num_samples)]
    return orig_img, augmented_samples


**Orquestación y Visualización de Resultados**

In [ ]:
def denormalize(tensor):
    # Revertir la normalización de ImageNet para mostrar la imagen correctamente
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    res = tensor * std + mean
    return clamp_np(res.permute(1, 2, 0).numpy())

def clamp_np(img_np):
    return np.clip(img_np, 0, 1)

# --- EJECUCIÓN DEL LABORATORIO ---
# Ruta ficticia: Reemplazar por una imagen local para pruebas ('test.jpg')
try:
    img_path = 'ejemplo.jpg'

    # Ejecutar OpenCV
    orig_gray, blur, edges = opencv_spatial_processing(img_path)

    # Ejecutar PyTorch
    pipeline = get_pytorch_augmentation_pipeline()
    orig_rgb, augmented_tensors = apply_pytorch_pipeline(img_path, pipeline)

    # --- PLOT 1: Filtros OpenCV ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig_gray, cmap='gray'); axes[0].set_title("Original (Gris)")
    axes[1].imshow(blur, cmap='gray'); axes[1].set_title("Desfoque Gaussiano 5x5")
    axes[2].imshow(edges, cmap='gray'); axes[2].set_title("Bordes Sobel Combinado")
    plt.tight_layout()
    plt.show()

    # --- PLOT 2: Aumentación PyTorch ---
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(orig_rgb); axes[0].set_title("Original (RGB)")
    for i, sample in enumerate(augmented_tensors):
        axes[i+1].imshow(denormalize(sample))
        axes[i+1].set_title(f"Aumentación #{i+1}")
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"⚠️ Nota de ejecución: {e}. Coloca una imagen válida en tu directorio de trabajo para ver los plots.")


**Integración con una CNN en PyTorch**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
import os

# 1. Configuración de Hardware Gratuito
# En Google Colab, asegúrate de ir a: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Entrenando en: {device}")

# 2. Creación del Dataset Personalizado
# Este objeto vincula tus imágenes en disco con las transformaciones de PyTorch
class LaboratorioDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Cargar imagen en RGB de forma perezosa (solo cuando se necesita)
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        # Aplicar el pipeline de Data Augmentation definido en el paso anterior
        if self.transform:
            image = self.transform(image)

        return image, label

# 3. Definición de la Red Neuronal (CNN)
# Usaremos una ResNet18 preentrenada (gratuita en torchvision.models)
def crear_modelo_cnn(num_clases=2):
    # Cargamos los pesos base para Transfer Learning
    modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Modificamos la última capa (Fully Connected) para ajustarla a nuestras clases
    num_features = modelo.fc.in_features
    modelo.fc = nn.Linear(num_features, num_clases)

    return modelo.to(device)

# 4. Pipeline de Entrenamiento Real
def entrenar_un_epoch(modelo, dataloader, optimizador, criterio):
    modelo.train() # Activa capas de Dropout y Batch Normalization
    loss_acumulada = 0.0

    # El DataLoader nos entrega lotes de imágenes transformadas al vuelo
    for imagenes, etiquetas in dataloader:
        # Mover datos a la GPU T4
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        # Resetear gradientes
        optimizador.zero_grad()

        # Forward Pass (Predicción)
        predicciones = modelo(imagenes)
        loss = criterio(predicciones, etiquetas)

        # Backward Pass y Optimización (Aprendizaje)
        loss.backward()
        optimizador.step()

        loss_acumulada += loss.item() * imagenes.size(0)

    return loss_acumulada / len(dataloader.dataset)

# --- SIMULACIÓN DE EJECUCIÓN ---
if __name__ == "__main__":
    # Creamos datos ficticios para demostrar la estructura (Reemplazar con tus rutas reales)
    # Por ejemplo, si tienes fotos de 'perros' y 'gatos'
    rutas_ficticias = ["imagen_01.jpg", "imagen_02.jpg", "imagen_03.jpg"]
    etiquetas_ficticias = [0, 1, 0] # Clase 0 o Clase 1

    # Crear archivos vacíos solo para evitar que falle el script si lo corres directo
    for r in rutas_ficticias:
        if not os.path.exists(r):
            Image.new('RGB', (300, 300), color='red').save(r)

    # 1. Recuperar el pipeline del bloque anterior
    pipeline_transformaciones = get_pytorch_augmentation_pipeline()

    # 2. Instanciar Dataset y DataLoader
    dataset_entrenamiento = LaboratorioDataset(
        image_paths=rutas_ficticias,
        labels=etiquetas_ficticias,
        transform=pipeline_transformaciones
    )

    # El DataLoader baraja los datos (shuffle) y usa hilos paralelos (num_workers)
    train_loader = DataLoader(dataset_entrenamiento, batch_size=2, shuffle=True, num_workers=0)

    # 3. Inicializar CNN, Optimizador y Función de Pérdida
    cnn_model = crear_modelo_cnn(num_clases=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

    # 4. Ejecutar 1 Época de prueba
    loss_resultado = entrenar_un_epoch(cnn_model, train_loader, optimizer, criterion)
    print(f"✅ Época completada exitosamente. Pérdida promedio (Loss): {loss_resultado:.4f}")

    # Limpieza de archivos ficticios
    for r in rutas_ficticias: os.remove(r)
